In [2]:
"""
02_silver.py — build the silver layer from bronze.
 
silver = enrich (canonicalize) + filter (high-confidence, mapped) + dedup.
 
Filter keeps a row only if BOTH signals are high and it mapped:
  - extraction_confidence == "high"  (bronze alignment guard)
  - match_confidence      == "high"  (dictionary alias hit)
  - metric_canonical is not None
 
Dedup collapses one row per (company, period, metric_canonical). The tiebreaker
prefers the value from the LATER-published report (restatements supersede
originals), falling back to ingest time only if report lineage isn't available.
 
Excluded rows are written to output/silver_excluded.csv so nothing is lost silently.
 
Run:  python 02_silver.py
"""
 
import json
from pathlib import Path
import pandas as pd
 
from canonical import canonicalize, CANONICAL_METRICS
 
BRONZE_DIR = Path("output/bronze")
OUT_DIR = Path("output/silver")
 
 
# ---------- load ----------
 
def load_bronze():
    records = []
    for f in sorted(BRONZE_DIR.glob("*.jsonl")):
        with open(f, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records
 
 
# ---------- enrich ----------
 
def enrich(record):
    r = canonicalize(record["metric"])
    canonical = r["canonical"]
    unit = CANONICAL_METRICS[canonical]["unit"] if canonical else record.get("unit")
    return {**record,
            "metric_canonical": r["canonical"],
            "unit": unit, 
            "match_method": r["method"],
            "match_confidence": r["confidence"]}

 
# ---------- build ----------
 
def build_silver(bronze_records):
    enriched = [enrich(r) for r in bronze_records]
    df = pd.DataFrame(enriched)
 
    # normalize the extraction-confidence column name (older bronze used "confidence")
    if "extraction_confidence" not in df.columns and "confidence" in df.columns:
        df = df.rename(columns={"confidence": "extraction_confidence"})
    if "extraction_confidence" not in df.columns:
        df["extraction_confidence"] = "high"   # default if bronze didn't tag it
 
    # filter: trustworthy AND mapped
    keep = (
        (df["extraction_confidence"] == "high")
        & (df["match_confidence"] == "high")
        & df["metric_canonical"].notna()
    )
    silver = df[keep].copy()
    excluded = df[~keep].copy()
 
    # dedup: restatement-aware. Prefer later-published report; fall back to ingest time.
    sort_keys = [k for k in ["report_period", "ingested_at"] if k in silver.columns]
    if sort_keys:
        silver = silver.sort_values(sort_keys)
    silver = silver.drop_duplicates(
        ["company", "period", "metric_canonical"], keep="last"
    ).reset_index(drop=True)
 
    return silver, excluded
 
 
# ---------- driver ----------
 
def main():
    bronze = load_bronze()
    print(f"bronze:   {len(bronze)} records")
 
    silver, excluded = build_silver(bronze)
    print(f"silver:   {len(silver)} kept")
    print(f"excluded: {len(excluded)} (low extraction / low match / unmapped)")
 
    OUT_DIR.mkdir(exist_ok=True)
    silver.to_csv(OUT_DIR / "silver.csv", index=False)
    excluded.to_csv(OUT_DIR / "silver_excluded.csv", index=False)
 
    if len(excluded):
        print("\nexcluded rows:")
        for _, r in excluded.iterrows():
            if pd.isna(r["metric_canonical"]):
                reason = "unmapped"
            elif r["extraction_confidence"] != "high":
                reason = "low extraction"
            else:
                reason = f"{r['match_confidence']} match"
            print(f"  {r['company']:12s} {r['metric']:34s} [{reason}]")
 
    print("\nsilver preview:")
    cols = ["company", "period", "metric_canonical", "value", "unit"]
    cols = [c for c in cols if c in silver.columns]
    print(silver[cols].to_string(index=False))
 
    print(f"\nwrote {OUT_DIR/'silver.csv'} and {OUT_DIR/'silver_excluded.csv'}")
 
 
if __name__ == "__main__":
    main()
 

bronze:   1197 records
silver:   130 kept
excluded: 657 (low extraction / low match / unmapped)

excluded rows:
  ApexFreight  Completed Shipments                [unmapped]
  ApexFreight  Active Shippers                    [unmapped]
  ApexFreight  Active Carriers                    [unmapped]
  ApexFreight  Recognized Revenue (transaction)   [medium match]
  ApexFreight  SaaS Tool Fee Revenue              [medium match]
  ApexFreight  Marketing Spend as % of Revenue    [medium match]
  ApexFreight  Support Tickets / 1,000 Shipments  [unmapped]
  ApexFreight  Completed Shipments                [unmapped]
  ApexFreight  Active Shippers                    [unmapped]
  ApexFreight  Active Carriers                    [unmapped]
  ApexFreight  Recognized Revenue (transaction)   [medium match]
  ApexFreight  SaaS Tool Fee Revenue              [medium match]
  ApexFreight  Marketing Spend as % of Revenue    [medium match]
  ApexFreight  Support Tickets / 1,000 Shipments  [unmapped]
  ApexFrei